# Project Additional Materials — Spatial Low Temperature Training

- Student ID: 10841269  
- Course Code: DATA70132  
- Academic Year: 2024–25  

**Environment:** See `README` and `ERP_Environment_2025.yaml`.  

**Reproduction:**  
1. First run `Spatial_dataset_prepare.ipynb` to generate the split datasets. All outputs are saved in the `splits_mm/` folder, which must be at the same level as this notebook.  
2. Then run this notebook top to bottom in the same directory. It reads the prepared `.mm` files from `splits_mm/` and trains models on the spatial subsets.  

This notebook trains and evaluates models for the low-temperature spatial subsets (`mid40`, `bottom30`) using both CV and validation splits.  
The trained models and evaluation results are saved for reproducibility.  



## Step 0 — Import required libraries

In [1]:
import json, pickle, numpy as np
from flaml import AutoML
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## Step 1 — Load split datasets from `splits_mm/`
Load the prepared `.mm` files (bottom30, mid40) generated by `Spatial_dataset_prepare.ipynb`.  
These datasets are stored inside the `splits_mm/` folder at the same level as this notebook.

In [2]:
SEED = 42
DTYPE = np.dtype("float32")

def load_split(prefix):
    with open(f"splits_mm/{prefix}_meta.json", "r") as f:
        meta = json.load(f)
    rows, cols = int(meta["rows"]), int(meta["cols"])
    X = np.memmap(f"splits_mm/{prefix}_X.mm", mode="r", dtype=DTYPE, shape=(rows, cols))
    y = np.memmap(f"splits_mm/{prefix}_y.mm", mode="r", dtype=DTYPE, shape=(rows,))
    return np.asarray(X), np.asarray(y), meta.get("features", None)

def print_metrics(tag, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    loss = 1.0 - r2
    print(f"[{tag}] R2={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, loss(1-R2)={loss:.4f}")
    return r2, rmse, mae, loss

# ===== Load splits: Train=bottom30, Val=mid40, Test=top30 =====
X_train, y_train, feats_train = load_split("bottom30")
X_val,   y_val,   feats_val  = load_split("mid40")


## Step 2 — Train Baseline AutoML model with FLAML (CV)
Train the models and select the best one to save `automl_low2high_cv.pkl`.

In [3]:
# ===== FLAML training with CV (bottom30 only) =====
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="regression",
    metric="r2",
    time_budget=3600,
    eval_method="cv",
    n_splits=5,
    seed=42,
    verbose=1,
)

print("[CV] Best estimator:", automl.best_estimator)
print("[CV] Best config:", automl.best_config)
print("[CV] Best CV loss (1 - R2):", automl.best_loss)

# save model
model_save_path = "automl_low2high_cv.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[CV] Model saved as {model_save_path}")

[CV] Best estimator: catboost
[CV] Best config: {'early_stopping_rounds': 10, 'learning_rate': 0.09999999999999996, 'n_estimators': 8192}
[CV] Best CV loss (1 - R2): 0.019486881639869714
[CV] Model saved as automl_low2high_cv.pkl


## Step 3 — Train Baseline AutoML model with FLAML (Mid40)
Train the models and select the best one to save `automl_low2high_val.pkl`.

In [4]:
# ===== FLAML training with explicit validation (bottom30 train, mid40 val) =====
automl = AutoML()
automl.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,   # external validation
    task="regression",
    metric="r2",
    time_budget=3600,
    eval_method="holdout",
    seed=42,
    verbose=1,
)

print("[VAL] Best estimator:", automl.best_estimator)
print("[VAL] Best config:", automl.best_config)
# predict on validation set
y_val_pred = automl.predict(X_val)
print_metrics("VAL(mid40)", y_val, y_val_pred)

# save model
model_save_path = "automl_low2high_val.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[VAL] Model saved as {model_save_path}")


[VAL] Best estimator: rf
[VAL] Best config: {'n_estimators': 93, 'max_features': 0.31862928165290877, 'max_leaves': 1383}
[VAL(mid40)] R2=0.9386, RMSE=1.2397, MAE=0.9671, loss(1-R2)=0.0614
[VAL] Model saved as automl_low2high_val.pkl


## Step 4 — Train Random Forest AutoML model with FLAML (CV)
Train the models and select the best one to save `automl_rf_low2high_cv.pkl`.

In [5]:
# ===== RF + CV (bottom30 only) =====
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="regression",
    metric="r2",
    time_budget=3600,
    eval_method="cv",
    n_splits=5,
    estimator_list=["rf"],
    n_jobs=-1,
    seed=42,
    verbose=1,
)

print("[RF-CV] Best estimator:", automl.best_estimator)
print("[RF-CV] Best config:", automl.best_config)
print("[RF-CV] Best CV loss (1 - R2):", automl.best_loss)

model_save_path = "automl_rf_low2high_cv.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[RF-CV] Model saved as {model_save_path}")

[RF-CV] Best estimator: rf
[RF-CV] Best config: {'n_estimators': 4, 'max_features': 1.0, 'max_leaves': 29}
[RF-CV] Best CV loss (1 - R2): 0.07501078230113892
[RF-CV] Model saved as automl_rf_low2high_cv.pkl


## Step 5 — Train Random Forest AutoML model with FLAML (Mid40)
Train the models and select the best one to save `automl_rf_low2high_val.pkl`.

In [6]:
# ===== RF + explicit validation (mid40) =====
automl = AutoML()
automl.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    task="regression",
    metric="r2",
    time_budget=3600,
    eval_method="holdout",
    estimator_list=["rf"],
    n_jobs=-1,
    seed=42,
    verbose=1,
)

print("[RF-VAL] Best estimator:", automl.best_estimator)
print("[RF-VAL] Best config:", automl.best_config)

# predict on validation set
y_val_pred = automl.predict(X_val)
print_metrics("VAL(mid40)", y_val, y_val_pred)

# save model
model_save_path = "automl_rf_low2high_val.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[RF-VAL] Model saved as {model_save_path}")


[RF-VAL] Best estimator: rf
[RF-VAL] Best config: {'n_estimators': 135, 'max_features': 0.33894560535479695, 'max_leaves': 1609}
[VAL(mid40)] R2=0.9389, RMSE=1.2369, MAE=0.9646, loss(1-R2)=0.0611
[RF-VAL] Model saved as automl_rf_low2high_val.pkl


## Step 6 — Train XGBoost AutoML model with FLAML (CV)
Train the models and select the best one to save `automl_xgb_low2high_cv.pkl`.

In [7]:
# ===== XGBoost + CV (bottom30 only) =====
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="regression",
    metric="r2",
    time_budget=3600,
    eval_method="cv",
    n_splits=5,
    estimator_list=["xgboost"],
    n_jobs=-1,
    seed=42,
    verbose=1,
)

print("[XGB-CV] Best estimator:", automl.best_estimator)
print("[XGB-CV] Best config:", automl.best_config)
print("[XGB-CV] Best CV loss (1 - R2):", automl.best_loss)

model_save_path = "automl_xgb_low2high_cv.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[XGB-CV] Model saved as {model_save_path}")


[XGB-CV] Best estimator: xgboost
[XGB-CV] Best config: {'n_estimators': 1109, 'max_leaves': 18, 'min_child_weight': 0.40385496411102617, 'learning_rate': 0.0951546340177734, 'subsample': 0.7621325607358561, 'colsample_bylevel': 0.896142769508154, 'colsample_bytree': 0.9993271961638156, 'reg_alpha': 0.0014585172191691578, 'reg_lambda': 17.50258170562381}
[XGB-CV] Best CV loss (1 - R2): 0.01917603152932985
[XGB-CV] Model saved as automl_xgb_low2high_cv.pkl


## Step 7 — Train XGBoost AutoML model with FLAML (Mid40)
Train the models and select the best one to save `automl_xgb_low2high_val.pkl`.

In [8]:
# ===== XGBoost + explicit validation (mid40) =====
automl = AutoML()
automl.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    task="regression",
    metric="r2",
    time_budget=3600,
    eval_method="holdout",
    estimator_list=["xgboost"],
    n_jobs=-1,
    seed=42,
    verbose=1,
)

print("[XGB-VAL] Best estimator:", automl.best_estimator)
print("[XGB-VAL] Best config:", automl.best_config)

# predict on validation set
y_val_pred = automl.predict(X_val)
print_metrics("VAL(mid40)", y_val, y_val_pred)

# save model
model_save_path = "automl_xgb_low2high_val.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[XGB-VAL] Model saved as {model_save_path}")


[XGB-VAL] Best estimator: xgboost
[XGB-VAL] Best config: {'n_estimators': 278, 'max_leaves': 8, 'min_child_weight': 49.33020408687341, 'learning_rate': 0.05357041879779343, 'subsample': 0.819529066861894, 'colsample_bylevel': 0.9824825588599275, 'colsample_bytree': 0.9200169452934103, 'reg_alpha': 0.02953751139763348, 'reg_lambda': 2.4702700632580834}
[VAL(mid40)] R2=0.9350, RMSE=1.2755, MAE=0.9889, loss(1-R2)=0.0650
[XGB-VAL] Model saved as automl_xgb_low2high_val.pkl


## Step 8 — Train LightGBM AutoML model with FLAML (CV)
Train the models and select the best one to save `automl_lgbm_low2high_cv.pkl`.

In [9]:
# ===== LightGBM + CV (bottom30 only) =====
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="regression",
    metric="r2",
    time_budget=3600,
    eval_method="cv",
    n_splits=5,
    estimator_list=["lgbm"],
    n_jobs=-1,
    seed=42,
    verbose=1,
)

print("[LGBM-CV] Best estimator:", automl.best_estimator)
print("[LGBM-CV] Best config:", automl.best_config)
print("[LGBM-CV] Best CV loss (1 - R2):", automl.best_loss)

model_save_path = "automl_lgbm_low2high_cv.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[LGBM-CV] Model saved as {model_save_path}")

[LGBM-CV] Best estimator: lgbm
[LGBM-CV] Best config: {'n_estimators': 2133, 'num_leaves': 43, 'min_child_samples': 13, 'learning_rate': 1.0, 'log_max_bin': 7, 'colsample_bytree': 1.0, 'reg_alpha': 0.021905145457011655, 'reg_lambda': 0.1934421673612076}
[LGBM-CV] Best CV loss (1 - R2): 0.01930803324621433
[LGBM-CV] Model saved as automl_lgbm_low2high_cv.pkl


## Step 9 — Train LightGBM AutoML model with FLAML (Mid40)
Train the models and select the best one to save `automl_lgbm_low2high_val.pkl`.

In [3]:
# ===== LightGBM + explicit validation (mid40) =====
automl = AutoML()
automl.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    task="regression",
    metric="r2",
    time_budget=3600,
    eval_method="holdout",
    estimator_list=["lgbm"],
    n_jobs=-1,
    seed=42,
    verbose=1,
)

print("[LGBM-VAL] Best estimator:", automl.best_estimator)
print("[LGBM-VAL] Best config:", automl.best_config)

# predict on validation set
y_val_pred = automl.predict(X_val)
print_metrics("VAL(mid40)", y_val, y_val_pred)

# save model
model_save_path = "automl_lgbm_low2high_val.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[LGBM-VAL] Model saved as {model_save_path}")


[LGBM-VAL] Best estimator: lgbm
[LGBM-VAL] Best config: {'n_estimators': 346, 'num_leaves': 10, 'min_child_samples': 45, 'learning_rate': 0.054144692789253865, 'log_max_bin': 9, 'colsample_bytree': 0.6988410109564447, 'reg_alpha': 0.2907119084485256, 'reg_lambda': 0.012845395662313684}
[VAL(mid40)] R2=0.9339, RMSE=1.2862, MAE=0.9959, loss(1-R2)=0.0661
[LGBM-VAL] Model saved as automl_lgbm_low2high_val.pkl
